In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_de;
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_de_raw_landing;

In [0]:
import csv
import json
import random
from datetime import datetime, timedelta, timezone
from pathlib import Path

BASE_DATA_DIR = Path("/Volumes/workspace/ecommerce_de/raw_landing")


def generate_batch(batch_id: str, num_orders: int = 500, inject_drift: bool = False):
    for sub in ["orders_csv", "payments_json", "customers_snapshot"]:
        (BASE_DATA_DIR / sub).mkdir(parents=True, exist_ok=True)

    now = datetime.now(timezone.utc)
    run_tag = now.strftime("%Y%m%d_%H%M%S")

    customer_ids = [f"CUST_{i:04d}" for i in range(1, 101)]
    statuses = ["COMPLETED", "PENDING", "CANCELLED", "REFUNDED"]
    payment_methods = ["CARD", "UPI", "NETBANKING", "WALLET"]

    orders_rows = []
    payment_events = []

    for i in range(1, num_orders + 1):
        order_id = f"ORD_{batch_id}_{i:05d}"
        cust_id = random.choice(customer_ids)

        # Late-arriving event (~10% chance, 1 to 3 days old)
        is_late = random.random() < 0.10
        event_ts = now - timedelta(
            days=random.randint(1, 3) if is_late else 0,
            minutes=random.randint(1, 600),
        )

        # Messy amounts & edge cases
        r = random.random()
        if r < 0.03:
            amount = None  # null amount -> quarantine
        elif r < 0.05:
            amount = -abs(round(random.uniform(10, 500), 2))  # negative amount -> quarantine
        elif r < 0.08:
            amount = f"{round(random.uniform(20, 2000), 2)}"  # stringified number
        else:
            amount = round(random.uniform(50, 5000), 2)

        bad_order_id = None if random.random() < 0.02 else order_id
        status = "UNKNOWN_STATUS" if random.random() < 0.02 else random.choice(statuses)

        row = {
            "order_id": bad_order_id,
            "customer_id": cust_id,
            "order_amount": amount,
            "currency": "INR",
            "status": status,
            "created_at": event_ts.isoformat(),
            "batch_id": batch_id,
        }

        if inject_drift:
            row["promo_code"] = random.choice(["", "SAVE10", "NEWUSER"])

        orders_rows.append(row)

        # Duplicate order (~4%)
        if bad_order_id and random.random() < 0.04:
            orders_rows.append(dict(row))

        # Payment event (JSONL)
        pay_event = {
            "event_id": f"PAY_{batch_id}_{i:05d}",
            "order_id": order_id,
            "customer_id": cust_id,
            "event_timestamp": event_ts.isoformat(),
            "payment": {
                "method": random.choice(payment_methods),
                "status": "SUCCESS" if status == "COMPLETED" else "FAILED",
                "amount": amount if isinstance(amount, (int, float)) else 0.0,
            },
            "batch_id": batch_id,
        }
        payment_events.append(pay_event)

        # Duplicate payment event (~3%)
        if random.random() < 0.03:
            payment_events.append(dict(pay_event))

    # Write Orders CSV
    orders_path = BASE_DATA_DIR / f"orders_csv/orders_{run_tag}_{batch_id}.csv"
    with open(orders_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(orders_rows[0].keys()))
        writer.writeheader()
        writer.writerows(orders_rows)

    # Write Payments JSONL
    payments_path = BASE_DATA_DIR / f"payments_json/payments_{run_tag}_{batch_id}.jsonl"
    with open(payments_path, "w", encoding="utf-8") as f:
        for ev in payment_events:
            f.write(json.dumps(ev) + "\n")

    # Write Customers Snapshot CSV (for SCD Type 2 testing)
    tiers = ["SILVER", "GOLD", "PLATINUM"]
    cities = ["Lucknow", "Bengaluru", "Delhi", "Mumbai", "Pune"]
    cust_path = BASE_DATA_DIR / f"customers_snapshot/customers_{run_tag}_{batch_id}.csv"
    with open(cust_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f, fieldnames=["customer_id", "city", "loyalty_tier", "updated_at"]
        )
        writer.writeheader()
        for cid in customer_ids:
            writer.writerow(
                {
                    "customer_id": cid,
                    "city": random.choice(cities),
                    "loyalty_tier": random.choice(tiers),
                    "updated_at": now.isoformat(),
                }
            )

    print(
        f"✅ Batch {batch_id} generated -> Orders: {len(orders_rows)}, Payments: {len(payment_events)}, Customers: {len(customer_ids)}"
    )


generate_batch("B001", num_orders=500, inject_drift=False)
generate_batch("B002", num_orders=500, inject_drift=True)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/ecommerce_de/raw_landing/orders_csv"))